# Giai đoạn 2 — Xây dựng đặc trưng mở rộng

Notebook này mở rộng Mục V của tài liệu gốc, dựa trên phát hiện ở
[01_deep_eda.ipynb](01_deep_eda.ipynb): gộp chung `has_promo` che mất tín hiệu thật, vì
`promo_channel` và `stackable_flag` có tác dụng rất khác nhau (`social_media` +8.3%,
`stackable=1` +6.1%, trong khi `in_store`/`all_channels` lại âm sâu).

Bốn nhóm đặc trưng mới, cộng thêm bộ 53 đặc trưng gốc:

| Nhóm | Nội dung | Vì sao hợp lệ (không rò rỉ) |
|---|---|---|
| A. Khuyến mãi mở rộng | `promo_channel` (one-hot), `promo_type`, `stackable_flag`, `min_order_value`, số ngày còn lại tới khi hết khuyến mãi | Lịch khuyến mãi công bố trước — giống lý do Mục V gốc giữ nhóm khuyến mãi |
| B. Lag của giá vốn (COGS) | `cogs` lùi ≥ H ngày, tương tự nhóm lịch sử doanh thu | COGS cùng ngày là rò rỉ (Mục V.5 gốc chứng minh), nhưng COGS đã lùi ≥ H ngày thì hợp lệ y hệt lag doanh thu |
| C. Ngày lễ lớn riêng biệt | đếm ngược tới 11/11 và 12/12 thay vì gộp chung `is_special_day` | Lịch dương biết trước tuyệt đối |
| D. Tương tác | `promo_channel × is_weekend`, `promo_any × is_weekend` | Suy ra từ hai nhóm cột đã hợp lệ ở trên |

**Kỷ luật giữ nguyên từ tài liệu gốc:** mỗi nhóm liên quan tới lịch sử một chuỗi số (COGS)
phải qua lại phép thử phá hoại ở Mục V.3; và **không nhóm nào được giữ lại chỉ vì nghe hợp
lý** — mọi nhóm phải chứng minh cải thiện WAPE bằng kiểm định chéo (Fold 2, Fold 3) trước
khi đưa vào bộ đặc trưng cuối cùng, y hệt cách Mục VI.5 gốc kiểm chứng bốn nhóm đặc trưng
ban đầu.

## 1. Thiết lập

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

H = 28
TARGET = "revenue"
SEED = 42

FOLDS = [("Fold 1", "2019-05-31", "2019-07-01", "2019-12-31"),
         ("Fold 2", "2020-05-31", "2020-07-01", "2020-12-31"),
         ("Fold 3", "2021-05-31", "2021-07-01", "2021-12-31")]
TEST_FOLD = ("TEST", "2022-05-31", "2022-07-01", "2022-12-31")
CV_FOLDS = ["Fold 2", "Fold 3"]

TET_DATES = ["2012-01-23", "2013-02-10", "2014-01-31", "2015-02-19", "2016-02-08",
             "2017-01-28", "2018-02-16", "2019-02-05", "2020-01-25", "2021-02-12",
             "2022-02-01", "2023-01-22"]
RETAIL_SPECIAL_DAYS = [(1, 1), (3, 8), (4, 30), (5, 1), (9, 2),
                        (10, 20), (11, 11), (12, 12), (12, 24), (12, 25)]

raw = pd.read_csv(DATA_PROCESSED / "sales_clean.csv", parse_dates=["date"]).sort_values("date")
raw = raw.reset_index(drop=True)
promotions = pd.read_csv(DATA_RAW / "promotions.csv", parse_dates=["start_date", "end_date"])
print(f"sales_clean: {len(raw):,} dòng   promotions: {len(promotions)} chương trình")
print("promo_type :", sorted(promotions.promo_type.unique()))
print("promo_channel:", sorted(promotions.promo_channel.unique()))

sales_clean: 3,833 dòng   promotions: 50 chương trình
promo_type : ['fixed', 'percentage']
promo_channel: ['all_channels', 'email', 'in_store', 'online', 'social_media']


## 2. Dựng lại bộ đặc trưng gốc (53 cột, Mục V tài liệu)

Sao chép nguyên trạng bốn hàm đặc trưng gốc để làm nền so sánh — mọi nhóm mới ở dưới đều
được đo bằng "thêm bao nhiêu so với nền 53 cột này", không phải so với con số tuyệt đối.

In [2]:
def add_lag_features(df, col=TARGET):
    """Đặc trưng từ lịch sử doanh thu. Mọi cột đều đã lùi ít nhất H ngày."""
    y = df[col]
    safe = y.shift(H)

    for lag in (28, 35, 42, 56, 91, 182, 364, 371):
        df[f"lag_{lag}"] = y.shift(lag)

    for w in (7, 28, 91, 364):
        df[f"roll_mean_{w}"] = safe.rolling(w, min_periods=max(2, w // 4)).mean()
        df[f"roll_std_{w}"] = safe.rolling(w, min_periods=max(2, w // 4)).std()
    df["roll_min_91"] = safe.rolling(91, min_periods=20).min()
    df["roll_max_91"] = safe.rolling(91, min_periods=20).max()

    df["lag364_smooth7"] = y.shift(364 - 3).rolling(7, min_periods=3).mean()
    df["lag364_smooth28"] = y.shift(364 - 14).rolling(28, min_periods=10).mean()

    df["ratio_28_364"] = df.roll_mean_28 / df.roll_mean_364
    df["ratio_91_364"] = df.roll_mean_91 / df.roll_mean_364
    df["trend_28_91"] = df.roll_mean_28 - df.roll_mean_91
    df["cv_91"] = df.roll_std_91 / df.roll_mean_91
    return df


def add_calendar_features(df):
    d = df["date"]
    df["dow"] = d.dt.dayofweek
    df["dom"] = d.dt.day
    df["month"] = d.dt.month
    df["quarter"] = d.dt.quarter
    df["doy"] = d.dt.dayofyear
    df["days_in_month"] = d.dt.days_in_month
    df["dom_reverse"] = df.days_in_month - df.dom
    df["is_weekend"] = (df.dow >= 5).astype("int8")
    df["is_eom"] = (df.dom_reverse <= 3).astype("int8")
    df["is_som"] = (df.dom <= 2).astype("int8")
    df["is_odd_year"] = (d.dt.year % 2).astype("int8")
    for k in (1, 2, 3, 4):
        df[f"fourier_sin{k}"] = np.sin(2 * np.pi * k * df.doy / 365.25)
        df[f"fourier_cos{k}"] = np.cos(2 * np.pi * k * df.doy / 365.25)
    return df


def add_tet_features(df):
    d = df["date"]
    tet = pd.to_datetime(pd.Series(TET_DATES))
    diff = np.stack([(d - t).dt.days.values for t in tet])
    df["days_to_tet"] = diff[np.abs(diff).argmin(axis=0), np.arange(len(df))]
    df["tet_before"] = df.days_to_tet.between(-30, -1).astype("int8")
    df["tet_after"] = df.days_to_tet.between(0, 14).astype("int8")
    sp = set(RETAIL_SPECIAL_DAYS)
    df["is_special_day"] = np.array([(m, dd) in sp for m, dd in
                                      zip(d.dt.month, d.dt.day)], dtype="int8")
    return df


def add_promo_features(df, promotions=promotions):
    df["promo_n"], df["promo_disc_max"], df["promo_day_idx"] = 0, 0.0, -1
    df["promo_streetwear"], df["promo_outdoor"] = 0, 0
    for _, p in promotions.iterrows():
        m = (df.date >= p.start_date) & (df.date <= p.end_date)
        if not m.any():
            continue
        df.loc[m, "promo_n"] += 1
        df.loc[m, "promo_disc_max"] = np.maximum(df.loc[m, "promo_disc_max"],
                                                   float(p.discount_value))
        idx = (df.loc[m, "date"] - p.start_date).dt.days
        cur = df.loc[m, "promo_day_idx"]
        df.loc[m, "promo_day_idx"] = np.where(cur < 0, idx, np.minimum(cur, idx))
        if p.applicable_category == "Streetwear":
            df.loc[m, "promo_streetwear"] = 1
        elif p.applicable_category == "Outdoor":
            df.loc[m, "promo_outdoor"] = 1
    df["promo_any"] = (df.promo_n > 0).astype("int8")
    return df


BASE_GROUP_PREFIXES = {
    "lịch sử doanh thu": ("lag", "roll", "ratio", "trend", "cv_"),
    "lịch": ("dow", "dom", "month", "quarter", "doy", "days_in", "is_week",
             "is_eom", "is_som", "is_odd", "fourier"),
    "Tết và ngày đặc biệt": ("days_to_tet", "tet_", "is_special"),
    "khuyến mãi": ("promo",),
}

base = add_promo_features(add_tet_features(add_calendar_features(add_lag_features(raw.copy()))))
base_features = [c for c in base.columns if c not in ("date", TARGET, "cogs")]
print(f"Bộ đặc trưng gốc: {len(base_features)} cột (tài liệu gốc: 53 cột)")
assert len(base_features) == 53, "lệch so với tài liệu gốc, kiểm tra lại"

Bộ đặc trưng gốc: 53 cột (tài liệu gốc: 53 cột)


## 3. Nhóm mới A — Khuyến mãi mở rộng

`add_promo_features` gốc chỉ giữ `promo_n`, `promo_disc_max`, `promo_day_idx`,
`promo_streetwear/outdoor`, `promo_any` — bỏ qua `promo_type`, `promo_channel`,
`stackable_flag`, `min_order_value`. Khi nhiều khuyến mãi chồng ngày, các cột phân loại
mới lấy theo chương trình có `discount_value` lớn nhất đang chạy (nhất quán với cách
`promo_disc_max` đã chọn), giống cách `promo_streetwear/outdoor` gốc xử lý chồng lấn.

In [3]:
CHANNELS = sorted(promotions.promo_channel.unique())     # email, online, social_media, in_store, all_channels


def add_promo_features_v2(df, promotions=promotions):
    df = add_promo_features(df, promotions)

    df["promo_is_percentage"] = 0
    df["stackable_flag"] = 0
    df["min_order_value"] = 0.0
    df["promo_days_left"] = -1
    for ch in CHANNELS:
        df[f"promo_ch_{ch}"] = 0

    best_discount = pd.Series(-1.0, index=df.index)
    for _, p in promotions.iterrows():
        m = (df.date >= p.start_date) & (df.date <= p.end_date)
        if not m.any():
            continue
        take = m & (p.discount_value > best_discount)
        df.loc[take, "promo_is_percentage"] = int(p.promo_type == "percentage")
        df.loc[take, "stackable_flag"] = int(p.stackable_flag)
        df.loc[take, "min_order_value"] = float(p.min_order_value)
        df.loc[take, "promo_days_left"] = (p.end_date - df.loc[take, "date"]).dt.days
        best_discount.loc[take] = p.discount_value

        # kênh không loại trừ lẫn nhau theo cách promo_disc_max chọn (một kênh cho một
        # chương trình), nên đánh dấu độc lập cho mọi chương trình đang chạy chứ không
        # chỉ chương trình có discount lớn nhất
        df.loc[m, f"promo_ch_{p.promo_channel}"] = 1
    return df


ext = add_promo_features_v2(base.copy())
new_promo_cols = ["promo_is_percentage", "stackable_flag", "min_order_value",
                   "promo_days_left"] + [f"promo_ch_{c}" for c in CHANNELS]
print(f"Thêm {len(new_promo_cols)} cột: {new_promo_cols}")
ext[["date"] + new_promo_cols].loc[ext.promo_any == 1].head(5)

Thêm 9 cột: ['promo_is_percentage', 'stackable_flag', 'min_order_value', 'promo_days_left', 'promo_ch_all_channels', 'promo_ch_email', 'promo_ch_in_store', 'promo_ch_online', 'promo_ch_social_media']


,date,promo_is_percentage,stackable_flag,min_order_value,promo_days_left,promo_ch_all_channels,promo_ch_email,promo_ch_in_store,promo_ch_online,promo_ch_social_media
211,2013-01-31,1,0,0.0,29,0,0,1,0,0
212,2013-02-01,1,0,0.0,28,0,0,1,0,0
213,2013-02-02,1,0,0.0,27,0,0,1,0,0
214,2013-02-03,1,0,0.0,26,0,0,1,0,0
215,2013-02-04,1,0,0.0,25,0,0,1,0,0


## 4. Nhóm mới B — Lag của giá vốn (COGS)

COGS **cùng ngày** là rò rỉ (Mục V.5 gốc: tương quan 0.976 với doanh thu, chỉ biết được
sau khi bán xong). Nhưng COGS đã lùi ≥ H ngày thì hợp lệ y hệt lag doanh thu — cùng một
lý do, cùng một quy tắc `shift(H)` làm nền an toàn. Biên lợi nhuận lùi (`margin_lag_28`)
là một tỉ số phái sinh, tương tự cách `ratio_28_364` phái sinh từ lag doanh thu.

In [4]:
def add_cogs_lag_features(df):
    c = df["cogs"]
    y = df[TARGET]
    safe = c.shift(H)

    for lag in (28, 91, 364):
        df[f"cogs_lag_{lag}"] = c.shift(lag)

    df["cogs_roll_mean_28"] = safe.rolling(28, min_periods=7).mean()
    df["cogs_roll_mean_91"] = safe.rolling(91, min_periods=20).mean()

    # biên lợi nhuận lùi 28 ngày: 1 - cogs/revenue tại cùng một mốc quá khứ hợp lệ
    df["margin_lag_28"] = 1 - (c.shift(28) / y.shift(28))
    return df


ext = add_cogs_lag_features(ext)
new_cogs_cols = ["cogs_lag_28", "cogs_lag_91", "cogs_lag_364",
                  "cogs_roll_mean_28", "cogs_roll_mean_91", "margin_lag_28"]
print(f"Thêm {len(new_cogs_cols)} cột: {new_cogs_cols}")
ext[["date"] + new_cogs_cols].tail(3)

Thêm 6 cột: ['cogs_lag_28', 'cogs_lag_91', 'cogs_lag_364', 'cogs_roll_mean_28', 'cogs_roll_mean_91', 'margin_lag_28']


,date,cogs_lag_28,cogs_lag_91,cogs_lag_364,cogs_roll_mean_28,cogs_roll_mean_91,margin_lag_28
3830,2022-12-29,1683082.90,4023586.93,3948015.94,1.576647e+06,2.021873e+06,-0.008773
3831,2022-12-30,726739.78,2332362.64,3121467.33,1.570159e+06,2.010664e+06,-0.008827
3832,2022-12-31,751384.00,1293984.10,3019583.50,1.540113e+06,2.004420e+06,0.003078


## 5. Nhóm mới C — Ngày lễ lớn riêng biệt (11/11, 12/12)

`is_special_day` gốc gộp 10 ngày mốc bán lẻ vào một cờ nhị phân duy nhất, coi 11/11 và
12/12 (hai ngày hội mua sắm thương mại điện tử lớn nhất) ngang với 30/4 hay 2/9. Trước khi
thêm đặc trưng riêng, kiểm tra thực nghiệm xem hai ngày này có thực sự nổi bật hơn mức
trung bình `is_special_day` hay không.

In [5]:
check = base.copy()
check["year_ratio"] = check[TARGET] / check.groupby(check.date.dt.year)[TARGET].transform("mean")
is_1111 = (check.date.dt.month == 11) & (check.date.dt.day == 11)
is_1212 = (check.date.dt.month == 12) & (check.date.dt.day == 12)
is_other_special = (check.is_special_day == 1) & ~is_1111 & ~is_1212

print(f"11/11            : {check.loc[is_1111, 'year_ratio'].mean():.3f}  ({is_1111.sum()} ngày)")
print(f"12/12            : {check.loc[is_1212, 'year_ratio'].mean():.3f}  ({is_1212.sum()} ngày)")
print(f"Ngày đặc biệt khác: {check.loc[is_other_special, 'year_ratio'].mean():.3f}  ({is_other_special.sum()} ngày)")
print(f"Ngày thường       : {check.loc[check.is_special_day == 0, 'year_ratio'].mean():.3f}")

11/11            : 0.632  (11 ngày)
12/12            : 0.534  (11 ngày)
Ngày đặc biệt khác: 1.105  (84 ngày)
Ngày thường       : 1.000


In [6]:
def add_specific_holiday_features(df):
    d = df["date"]
    for name, month, day in [("1111", 11, 11), ("1212", 12, 12)]:
        marks = pd.to_datetime([f"{y}-{month:02d}-{day:02d}"
                                 for y in range(d.dt.year.min() - 1, d.dt.year.max() + 2)])
        diff = np.stack([(d - m).dt.days.values for m in marks])
        df[f"days_to_{name}"] = diff[np.abs(diff).argmin(axis=0), np.arange(len(df))]
    return df


ext = add_specific_holiday_features(ext)
new_holiday_cols = ["days_to_1111", "days_to_1212"]
print(f"Thêm {len(new_holiday_cols)} cột: {new_holiday_cols}")

Thêm 2 cột: ['days_to_1111', 'days_to_1212']


## 6. Nhóm mới D — Tương tác

Cây quyết định tự học được tương tác qua các lần chia nhánh liên tiếp, nhưng cho một
tương tác đã biết trước là quan trọng (Giai đoạn 1: `social_media` và cuối tuần cùng lúc),
một cột tường minh giúp mô hình không phải "dò" ra tương tác đó từ dữ liệu thưa.

In [7]:
ext["promo_weekend"] = ext.promo_any * ext.is_weekend
ext["promo_social_weekend"] = ext.get("promo_ch_social_media", 0) * ext.is_weekend
new_interaction_cols = ["promo_weekend", "promo_social_weekend"]
print(f"Thêm {len(new_interaction_cols)} cột: {new_interaction_cols}")

Thêm 2 cột: ['promo_weekend', 'promo_social_weekend']


## 7. Kiểm tra rò rỉ dữ liệu cho nhóm mới

Chỉ nhóm B (lag COGS) dẫn xuất từ một chuỗi lịch sử — cần chạy lại đúng phép thử phá hoại
ở Mục V.3 gốc, lần này phá hoại **cả hai** chuỗi (doanh thu và COGS) trong vùng bị chặn.
Ba nhóm còn lại (khuyến mãi mở rộng, ngày lễ, tương tác) là hàm xác định của ngày tháng và
lịch khuyến mãi đã công bố — không đọc `revenue`/`cogs` nên không thể rò rỉ, không cần
phép thử động.

In [8]:
def leakage_probe_cogs(raw, n_checks=8):
    fa = add_cogs_lag_features(add_lag_features(raw.copy()))
    cols = [c for c in fa.columns if c.startswith(("lag", "roll", "ratio", "trend", "cv_",
                                                     "cogs_lag", "cogs_roll", "margin_lag"))]
    rows = np.linspace(max(H, 400), len(raw) - 1, n_checks, dtype=int)
    worst = 0.0
    for row in rows:
        b = raw.copy()
        block = slice(row - H + 1, row + 1)
        b.loc[b.index[block], TARGET] *= 9.3
        b.loc[b.index[block], "cogs"] *= 9.3
        fb = add_cogs_lag_features(add_lag_features(b))
        worst = max(worst, float((fa[cols].iloc[row].fillna(-1)
                                   - fb[cols].iloc[row].fillna(-1)).abs().max()))
    return worst, len(rows), len(cols)


worst, n_checkpoints, n_columns = leakage_probe_cogs(raw)
print(f"Phá hoại {H} nhãn gần nhất (revenue + cogs) tại {n_checkpoints} mốc, kiểm {n_columns} cột")
print(f"Thay đổi lớn nhất quan sát được = {worst:.3e}")
assert worst < 1e-9, "RÒ RỈ: có đặc trưng nhìn thấy nhãn trong vùng bị chặn"
print("ĐẠT — không rò rỉ.")

Phá hoại 28 nhãn gần nhất (revenue + cogs) tại 8 mốc, kiểm 30 cột
Thay đổi lớn nhất quan sát được = 0.000e+00
ĐẠT — không rò rỉ.


## 8. Kiểm định: các nhóm mới có thực sự cải thiện WAPE không?

Cùng một mô hình (LightGBM, cấu hình giống hệt tài liệu gốc), cùng một cách chia tập —
chỉ khác bộ cột đưa vào. Quyết định giữ hay bỏ một nhóm dựa trên **CV** (trung bình Fold 2,
Fold 3), cột TEST chỉ hiển thị để tham khảo, không dùng để quyết định — đúng kỷ luật Mục
VI gốc.

In [9]:
def get_lightgbm(seed=SEED):
    import lightgbm as lgb
    return lgb.LGBMRegressor(n_estimators=700, learning_rate=0.04, subsample=0.85,
                              colsample_bytree=0.85, reg_lambda=1.0, n_jobs=-1,
                              random_state=seed, num_leaves=31, min_child_samples=20,
                              subsample_freq=5, verbose=-1)


def split_by_date(df, train_end, valid_start, valid_end):
    train = df[df.date <= train_end]
    valid = df[(df.date >= valid_start) & (df.date <= valid_end)]
    return train, valid


def metrics(y_true, y_pred):
    y = np.asarray(y_true, float)
    p = np.clip(np.asarray(y_pred, float), 0, None)
    return dict(WAPE=float(np.sum(np.abs(y - p)) / np.sum(y)),
                bias=float((p.mean() - y.mean()) / y.mean()),
                R2=float(1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)))


def evaluate_fold(df, fold, columns):
    train, valid = split_by_date(df, fold[1], fold[2], fold[3])
    model = get_lightgbm()
    model.fit(train[columns], np.log1p(train[TARGET].values))
    pred = np.expm1(model.predict(valid[columns]))
    return metrics(valid[TARGET].values, pred)["WAPE"]


def cross_validate(df, columns, folds=FOLDS, cv_folds=CV_FOLDS):
    scores = {f[0]: evaluate_fold(df, f, columns) for f in folds}
    scores["CV"] = float(np.mean([scores[f] for f in cv_folds]))
    return scores

In [10]:
ready = ext.dropna(subset=["lag_371", "roll_mean_364", "cogs_lag_364"]).reset_index(drop=True)
print(f"Sau khi bỏ giai đoạn khởi động (cần cả lag_371 và cogs_lag_364): {len(ready):,} dòng"
      f", từ {ready.date.min().date()} đến {ready.date.max().date()}")

FEATURE_SETS = {
    "A. gốc (53 cột)": base_features,
    "B. + khuyến mãi mở rộng": base_features + new_promo_cols,
    "C. + lag COGS": base_features + new_promo_cols + new_cogs_cols,
    "D. + ngày lễ riêng biệt": base_features + new_promo_cols + new_cogs_cols + new_holiday_cols,
    "E. + tương tác (đầy đủ)": (base_features + new_promo_cols + new_cogs_cols
                                 + new_holiday_cols + new_interaction_cols),
}

rows = []
for name, columns in FEATURE_SETS.items():
    scores = cross_validate(ready, columns)
    train, valid = split_by_date(ready, *TEST_FOLD[1:])
    model = get_lightgbm()
    model.fit(train[columns], np.log1p(train[TARGET].values))
    scores["TEST"] = metrics(valid[TARGET].values,
                              np.expm1(model.predict(valid[columns])))["WAPE"]
    rows.append({"set": name, "n_columns": len(columns), **scores})

table = pd.DataFrame(rows).set_index("set")
base_cv = table.loc["A. gốc (53 cột)", "CV"]
table["vs_A_pct"] = 100 * (table.CV / base_cv - 1)
print(table[["n_columns", "Fold 1", "Fold 2", "Fold 3", "CV", "vs_A_pct", "TEST"]]
      .round(4).rename_axis("bộ đặc trưng").to_string())

Sau khi bỏ giai đoạn khởi động (cần cả lag_371 và cogs_lag_364): 3,462 dòng, từ 2013-07-10 đến 2022-12-31


                         n_columns  Fold 1  Fold 2  Fold 3      CV  vs_A_pct    TEST
bộ đặc trưng                                                                        
A. gốc (53 cột)                 53  0.4904  0.2628  0.2172  0.2400    0.0000  0.2072
B. + khuyến mãi mở rộng         62  0.5156  0.2602  0.2181  0.2391   -0.3529  0.2008
C. + lag COGS                   68  0.4824  0.2561  0.2302  0.2431    1.3179  0.2069
D. + ngày lễ riêng biệt         70  0.4796  0.2673  0.2276  0.2474    3.0984  0.1991
E. + tương tác (đầy đủ)         72  0.4560  0.2604  0.2323  0.2464    2.6610  0.2022


**Đọc bảng:** `vs_A_pct` âm nghĩa là CV WAPE giảm (tốt hơn) so với bộ gốc. Chỉ giữ lại
những nhóm có đóng góp dương thực sự trên CV — một nhóm nghe hợp lý về mặt nghiệp vụ
(như tương tác `promo_social_weekend`) vẫn có thể không giúp ích nếu mô hình cây đã tự học
được tương tác đó qua cách chia nhánh, hoặc nếu số ngày dữ liệu quá ít để ước lượng ổn
định (`promo_ch_social_media` chỉ có 180 ngày trong 10 năm).

## 9. Chốt bộ đặc trưng cho Giai đoạn 3

In [11]:
best_set_name = table["CV"].idxmin()
final_columns = FEATURE_SETS[best_set_name]
print(f"Bộ đặc trưng tốt nhất theo CV: {best_set_name}  ({len(final_columns)} cột)")
print(f"CV WAPE = {table.loc[best_set_name, 'CV']:.4f}"
      f"  so với gốc {base_cv:.4f}  ({table.loc[best_set_name, 'vs_A_pct']:+.2f}%)")

final_groups = {}
for name, prefixes in BASE_GROUP_PREFIXES.items():
    cols = [c for c in base_features if c.startswith(prefixes)]
    final_groups[name] = {"n_columns": len(cols), "columns": cols}

candidate_new_groups = {"khuyến mãi mở rộng": new_promo_cols, "lag COGS": new_cogs_cols,
                         "ngày lễ riêng biệt": new_holiday_cols, "tương tác": new_interaction_cols}
for name, cols in candidate_new_groups.items():
    kept = [c for c in cols if c in final_columns]
    if kept:
        final_groups[name] = {"n_columns": len(kept), "columns": kept}

out_path = DATA_PROCESSED / f"features_h{H}_v2.csv"
ready[["date", TARGET, "cogs"] + final_columns].to_csv(out_path, index=False)
print(f"Đã ghi {out_path}  ({len(ready):,} dòng, {len(final_columns)} đặc trưng)")

import json
summary = {
    "ablation_table": table.round(6).to_dict("index"),
    "chosen_set": best_set_name,
    "final_features": final_columns,
    "n_final_features": len(final_columns),
    "new_groups": {"khuyến mãi mở rộng": new_promo_cols, "lag COGS": new_cogs_cols,
                   "ngày lễ riêng biệt": new_holiday_cols, "tương tác": new_interaction_cols},
}
(RESULTS / "02_features.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, default=lambda o: o.tolist()),
    encoding="utf-8")
print("Đã ghi results/02_features.json")

Bộ đặc trưng tốt nhất theo CV: B. + khuyến mãi mở rộng  (62 cột)
CV WAPE = 0.2391  so với gốc 0.2400  (-0.35%)


Đã ghi D:\Document\aio2026\module03\conquer\data\processed\features_h28_v2.csv  (3,462 dòng, 62 đặc trưng)
Đã ghi results/02_features.json


**Tổng kết Giai đoạn 2:** bộ đặc trưng cuối cùng và bảng ablation được lưu ở
`results/02_features.json`, dữ liệu đặc trưng đầy đủ ở `data/processed/features_h28_v2.csv`
— Giai đoạn 3 (modeling nâng cao: tuning có kỷ luật thời gian, baseline thống kê, dự báo
khoảng) sẽ đọc trực tiếp hai tệp này thay vì phải chạy lại bước xây dựng đặc trưng.